In [1]:
import requests
import pandas as pd
import time

In [2]:
API_key = "hpSobUtXBHcPw37mqPR1J3ER"
request = "https://www.searchapi.io/api/v1/search"

In [3]:
params = {
    "engine": "ebay_search",
    "q": "comics",
    "ebay_domain": "ebay.com",
    "num": 60,
    "page": 1
}

page = requests.get(request, headers={'Authorization': f'Bearer {API_key}'}, params=params)
page

<Response [200]>

In [4]:
res = page.json()
res.keys()

dict_keys(['search_metadata', 'search_parameters', 'search_information', 'categories', 'filters', 'organic_results', 'related_searches', 'pagination'])

In [5]:
res["organic_results"][0]

{'position': 1,
 'item_id': '295102178301',
 'title': 'LARGE 25 COMICS BOOK LOT-MARVEL, DC, INDIES- FREE/Fast Shipping! VF to NM+ ALL',
 'link': 'https://www.ebay.com/itm/295102178301',
 'seller': {'name': 'hadleycomics',
  'reviews': 1600,
  'positive_feedback_percent': 99.8},
 'condition': 'New (Other)',
 'buying_format': 'or Best Offer',
 'extensions': ['Free returns',
  '1,248 sold',
  'Save up to 10% when you buy more'],
 'price': '$22.49',
 'extracted_price': 22.49,
 'original_price': '$24.99',
 'extracted_original_price': 24.99,
 'shipping': 'Located in United States',
 'deal': 'Free delivery in 2-4 days',
 'thumbnail': 'https://i.ebayimg.com/images/g/EmQAAOSwmLFmq7~I/s-l500.jpg'}

In [6]:
def clean_page_columns(page_df):
    page_df['seller_name'] = page_df['seller'].apply(lambda d: d['name'] if pd.notna(d) and 'name' in d else None)
    page_df['seller_reviews'] = page_df['seller'].apply(lambda d: d['reviews'] if pd.notna(d) and 'reviews' in d else None)
    page_df['seller_feedback'] = page_df['seller'].apply(lambda d: d['positive_feedback_percent'] if pd.notna(d) and 'positive_feedback_percent' in d else None)
    page_df['extensions'] = page_df['extensions'].fillna('')
    page_df['sold_text'] = page_df['extensions'].apply(lambda x: x[2] if isinstance(x, list) and len(x) > 2 else None)
    page_df['return_info'] = page_df['extensions'].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
    page_df['location'] = page_df['extensions'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
    page_df.set_index('item_id', inplace=True)
    return page_df[['title', 'price', 'extracted_price', 'reviews', 'condition', 'buying_format', 'seller_name', 'seller_reviews', 'seller_feedback', 'sold_text', 'return_info', 'location', 'link']]

In [7]:
page_df = pd.DataFrame(res["organic_results"])
ebay_df_test = clean_page_columns(page_df)
ebay_df_test

,title,price,extracted_price,reviews,condition,buying_format,seller_name,seller_reviews,seller_feedback,sold_text,return_info,location,link
item_id,,,,,,,,,,,,,
295102178301,"LARGE 25 COMICS BOOK LOT-MARVEL, DC, INDIES- F...",$22.49,22.49,NaN,New (Other),or Best Offer,hadleycomics,1600,99.8,Save up to 10% when you buy more,"1,248 sold",Free returns,https://www.ebay.com/itm/295102178301
287039504394,DC Marvel Superman Spider-Man #1 (One Shot) Co...,$14.99,14.99,NaN,Brand New,Buy It Now,si-fi_toys,14400,99.6,None,187 sold,Free returns,https://www.ebay.com/itm/287039504394
284701338991,lot of 100 comics 80s 90s 2000 Marvel Dc Horse...,$28.95,28.95,NaN,Pre-Owned,Buy It Now,77kings,8200,99.9,None,"1,064 sold",Located in United States,https://www.ebay.com/itm/284701338991
227123160181,"Invincible Compendium 1, 2, & 3 Book Lot Compl...",$142.99,142.99,NaN,Brand New,Buy It Now,comicmaster4000,5800,99.2,None,93 sold,Located in United States,https://www.ebay.com/itm/227123160181
227289598146,Amazing Spider-Man (2025) #1-19 20 21 22 23 24...,$4.88 to $159.88,NaN,NaN,Brand New,Buy It Now,cosmic_kingdom_comics,15000,100.0,10% off $40+ with coupon,Free returns,Located in United States,https://www.ebay.com/itm/227289598146
336474481924,Absolute Batman 1 2 3 4 5 6 7 8 9 10 11 12 13 ...,$119.99,119.99,NaN,New (Other),Buy It Now,mccartycards,9100,99.8,None,61 sold,Located in United States,https://www.ebay.com/itm/336474481924
188257989469,Dead Samurai #6 Cover C Drax Gal Cover,$5.99,5.99,NaN,New (Other),or Best Offer,torpedocomics,7700,100.0,None,None,Located in United States,https://www.ebay.com/itm/188257989469
168293327141,Web Of Venom #1 (NM),$5.69,5.69,NaN,Brand New,Buy It Now,thecomicferret,13300,99.9,Almost gone,Free returns,Located in United States,https://www.ebay.com/itm/168293327141
156070129648,Godhood Comics The Antagonists NM-/M 2023,$119.95 to $149.95,NaN,NaN,Brand New,Buy It Now,multiversecomicsandmore,7200,100.0,None,None,Located in United States,https://www.ebay.com/itm/156070129648


In [8]:
queries = ["comics", "comic books", "graphic novels", "webcomics", "comic anthology"]

ebay_df = pd.DataFrame()

for query in queries:
    for page_num in range(1, 9):
        params = {
            "engine": "ebay_search",
            "q": query,
            "ebay_domain": "ebay.com",
            "num": 240,
            "page": page_num
        }

        page = requests.get(request, headers={'Authorization': f'Bearer {API_key}'}, params=params)

        if page.status_code != 200:
            break

        res = page.json()

        if "organic_results" not in res or len(res["organic_results"]) == 0:
            break

        page_df = pd.DataFrame(res["organic_results"])
        page_df = clean_page_columns(page_df)
        page_df["source_query"] = query
        page_df["source_page"] = page_num

        ebay_df = pd.concat([ebay_df, page_df])
        time.sleep(1)

In [9]:
ebay_df

,title,price,extracted_price,reviews,condition,buying_format,seller_name,seller_reviews,seller_feedback,sold_text,return_info,location,link,source_query,source_page
item_id,,,,,,,,,,,,,,,
295102178301,"LARGE 25 COMICS BOOK LOT-MARVEL, DC, INDIES- F...",$22.49,22.49,NaN,New (Other),or Best Offer,hadleycomics,1600,99.8,Save up to 10% when you buy more,"1,248 sold",Free returns,https://www.ebay.com/itm/295102178301,comics,1
376968283312,MARVEL/DC: SPIDER-MAN/SUPERMAN #1 *Dang! So Ma...,$7.89 to $89.99,NaN,NaN,Brand New,Buy It Now,odineyes_oddities,7800,100.0,20% off 20+ with coupon,Free returns,Located in United States,https://www.ebay.com/itm/376968283312,comics,1
188257989469,Dead Samurai #6 Cover C Drax Gal Cover,$5.99,5.99,NaN,New (Other),or Best Offer,torpedocomics,7700,100.0,None,None,Located in United States,https://www.ebay.com/itm/188257989469,comics,1
287039504394,DC Marvel Superman Spider-Man #1 (One Shot) Co...,$14.99,14.99,NaN,Brand New,Buy It Now,si-fi_toys,14400,99.6,None,187 sold,Free returns,https://www.ebay.com/itm/287039504394,comics,1
284701338991,lot of 100 comics 80s 90s 2000 Marvel Dc Horse...,$28.95,28.95,NaN,Pre-Owned,Buy It Now,77kings,8200,99.9,None,"1,064 sold",Located in United States,https://www.ebay.com/itm/284701338991,comics,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137208789148,Mirror Mirror 2,$27.77,27.77,NaN,Pre-Owned,Buy It Now,readingebooks,20100,99.7,None,None,Located in United States,https://www.ebay.com/itm/137208789148,comic anthology,8
406022997316,"Street Fighter 2 Comic Anthology, VG, no tears...",$99.73,99.73,NaN,New (Other),or Best Offer,kiyma87,314,99.4,None,None,Located in Japan,https://www.ebay.com/itm/406022997316,comic anthology,8
389350082654,Twilight Zone Comic #37 Gold Key May 1971 Vint...,$8.40,8.40,NaN,Pre-Owned,or Best Offer,nat_444992,24,100.0,None,None,Located in United States,https://www.ebay.com/itm/389350082654,comic anthology,8


In [ ]:
ebay_df.to_excel('ebay_comics.xlsx', index=True)